In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import regexp_replace, col, when, nullif
import matplotlib.pyplot as plt
import seaborn as sns
import os

# View available datasets

In [0]:
data_BASE_DIR = "dbfs:/mnt/mids-w261/"
display(dbutils.fs.ls(f"{data_BASE_DIR}/datasets_final_project_2022/"))

# Create checkpoint folder

In [0]:
section = "03"
number = "02"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"

# Size in rows/columns, GB of Weather Dataset

In [0]:
def get_shape(df):
  '''Return the shape of the dataframe'''
  return f"{df.count()} rows, {len(df.columns)} columns"

def get_size(path):
  total = 0
  for f in dbutils.fs.ls(path):
      if f.isDir():
          total += get_size(f.path)
      else:
          total += f.size
  return total

def get_null_rows(df):
  '''Display number of null records for each column in the dataframe.'''
  return df.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df.columns]).transpose()

def get_null_info(df):
    '''Display number of null records and null percentage for each column in the dataframe'''
    total_rows = df.count()

    exprs = [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]
    counts = df.select(exprs)

    long_df = counts.select(
        F.explode(F.map_from_arrays(
            F.array(*[F.lit(c) for c in df.columns]),
            F.array(*[F.col(c) for c in df.columns])
        )).alias("column", "null_count")
    ).withColumn("null_count", F.col("null_count").cast("long"))

    # Add percentage rounded to 2 decimals
    return long_df.withColumn(
        "null_pct",
        F.round((F.col("null_count") / total_rows) * 100, 2)
    )


In [0]:
# 1y and full weather data
df_weather_1y = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_1y/")
df_weather_full = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data/")

In [0]:
print(f"Shape of 1 year weather dataset: {get_shape(df_weather_1y)}")
print(f"Shape of full weather dataset: {get_shape(df_weather_full)}")

total_size_1y_gb = get_size(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_1y/") / (1024**3)
print(f"Size of 1 year weather dataset: {total_size_1y_gb:.2f} GB")

total_size_full_gb = get_size(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data/") / (1024**3)
print(f"Size of full weather dataset: {total_size_full_gb:.2f} GB")

# Ensure there are no duplicate rows in weather data

In [0]:
assert(df_weather_1y.count() == df_weather_1y.dropDuplicates().count())

# Count number of null records for each column in 1 year weather data

In [0]:
null_info_df_weather_1y = get_null_info(df_weather_1y)
display(null_info_df_weather_1y)

The weather dataset reports meterological observations at three time granularities: hourly, daily, and monthly. Because flights operate on a high-frequency schedule, we use the hourly observations as our primary weather features.

# Graphic summary of missing value analysis 

In [0]:
def get_weather_features_by_granularity(null_info_df):
    df_hourly_plot = null_info_df.filter(
        F.col("column").contains("Hourly")
    ).toPandas().sort_values("null_pct")

    df_daily_plot = null_info_df.filter(
        F.col("column").contains("Daily")
    ).toPandas().sort_values("null_pct")

    df_monthly_plot = null_info_df.filter(
        F.col("column").contains("Monthly")
    ).toPandas().sort_values("null_pct")

    return df_hourly_plot, df_daily_plot, df_monthly_plot

def graph_missing_values_by_granularity(null_info_df):
    df_hourly_plot, df_daily_plot, df_monthly_plot = get_weather_features_by_granularity(null_info_df)

    dfs = [
        ("Hourly Weather Features", df_hourly_plot),
        ("Daily Weather Features", df_daily_plot),
        ("Monthly Weather Features", df_monthly_plot)
    ]

    fig, axes = plt.subplots(3, 1, figsize=(12, 24))

    for ax, (title, df_plot) in zip(axes, dfs):

        if df_plot.empty:
            ax.set_title(f"{title} — No matching columns found", fontsize=14)
            ax.axis("off")
            continue

        sns.barplot(
            data=df_plot,
            x="null_pct",
            y="column",
            ax=ax
        )

        ax.set_title(f"Missing Value Percentage: {title} (1y dataset)", fontsize=16)
        ax.set_xlabel("Percentage of Missing Values (%)", fontsize=12)
        ax.set_ylabel("Feature", fontsize=12)

        # Annotate each bar
        for i, v in enumerate(df_plot["null_pct"]):
            ax.text(v + 0.5, i, f"{v:.2f}%", va="center")

    # plt.tight_layout()
    plt.show()


In [0]:
graph_missing_values_by_granularity(null_info_df_weather_1y)

# Filter for report type FM-15

In [0]:
df_weather_1y_fm15 = df_weather_1y.filter(F.col('REPORT_TYPE') == 'FM-15')
null_info_df_weather_1y_fm15 = get_null_info(df_weather_1y_fm15)

In [0]:
graph_missing_values_by_granularity(null_info_df_weather_1y_fm15)

In [0]:
# Show only rows with less than 50% null values, manually select which ones to keep
display(null_info_df_weather_1y_fm15.filter(F.col("null_pct") < 50))

In [0]:
weather_data_descriptors = ['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'REPORT_TYPE', 'YEAR']
weather_features_to_keep = [
    'HourlyAltimeterSetting', 'HourlyDryBulbTemperature', 'HourlyRelativeHumidity', 
    'HourlySkyConditions', 'HourlyStationPressure', 'HourlyVisibility', 
    'HourlyWindDirection', 'HourlyWindSpeed', 'HourlyPrecipitation'
]

# Get hourly weather features
df_weather_1y_hourly = df_weather_1y_fm15.select(weather_data_descriptors + weather_features_to_keep)# .dropna()
# display(df_weather_1y_hourly)

# Replace extraneous values using regex, then cast column to appropriate data type

In [0]:
hourly_cat_vars_to_regex_types = {
    'HourlyStationPressure': ['s', '', 'double'],
    'HourlyDryBulbTemperature': ["[^0-9\.-]", '', 'int'],
    'HourlyWindSpeed': ['s', '', 'int'],
    'HourlyVisibility': ['[^0-9\.]', '', 'double'],
    'HourlyRelativeHumidity': ['\*', '0', 'int'],
    'HourlyAltimeterSetting': ['s', '', 'double'],
    'HourlyWindDirection': ["[^0-9]", '', 'int'],
}

for col_name, payload in hourly_cat_vars_to_regex_types.items():
  regex_str, replace_with, cast_type = payload
  df_weather_1y_hourly = df_weather_1y_hourly.withColumn(col_name, regexp_replace(col_name, regex_str, replace_with).try_cast(cast_type))

# Handle HourlyPrecipitation cases separately

In [0]:
df_weather_1y_hourly = df_weather_1y_hourly.withColumn(
    "HourlyPrecipitation",
    F.regexp_replace(
        # Replace trace precipitation with 0.0001
        # Replace missing values with 0
        # Else leave as is, then remove 's' from end of string
        F.regexp_replace(
            F.when(F.col("HourlyPrecipitation").isin("T", "Ts"), '0.0001')
             .when(F.col('HourlyPrecipitation').isNull(), '0')
             .otherwise(F.col("HourlyPrecipitation")),
            "s$", ""
        ),
        # Remove all non-numeric characters
        "[^0-9.]", ""
    )
)

# 2. Drop corrupted concatenated values (multiple decimal points)
df_weather_1y_hourly = df_weather_1y_hourly.filter(
    ~F.col("HourlyPrecipitation").rlike(r"\d+\.\d+\.\d+")
)

# 3. Replace any remaining blank precipitation values with 0 and cast column to double
df_weather_1y_hourly = df_weather_1y_hourly.withColumn(
    "HourlyPrecipitation",
     F.when(F.col("HourlyPrecipitation") == "", '0')
     .otherwise(F.col("HourlyPrecipitation")).cast('double')
)

In [0]:
display(df_weather_1y_hourly.select('HourlyPrecipitation').distinct())

In [0]:
display(get_null_info(df_weather_1y_hourly))

In [0]:
# Verify all weather features are casted correctly
df_weather_1y_hourly.select(weather_features_to_keep).printSchema()

# Filter for only REPORT_TYPE FM15 (standard hourly observations). This is the most complete and consistent report type, making it ideal for linking to individual flights

# Forward fill hourly weather features then backwards-fill for leading nulls

In [0]:
display(df_weather_1y_hourly.select('HourlySkyConditions').distinct())

In [0]:
# Forward-fill window
w_ffill = (
    Window.partitionBy("STATION")
          .orderBy("DATE")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_weather_1y_hourly_fm15_ffill = df_weather_1y_hourly
for col in weather_features_to_keep:
    if col != 'HourlySkyConditions':
        df_weather_1y_hourly_fm15_ffill = df_weather_1y_hourly_fm15_ffill.withColumn(
            col,
            F.last(col, ignorenulls=True).over(w_ffill)
        )

# Backward-fill window
w_bfill = (
    Window.partitionBy("STATION")
          .orderBy(F.col("DATE").desc())
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_weather_1y_hourly_fm15_ffill_bfill = df_weather_1y_hourly_fm15_ffill
for col in weather_features_to_keep:
    if col != 'HourlySkyConditions':
        df_weather_1y_hourly_fm15_ffill_bfill = df_weather_1y_hourly_fm15_ffill_bfill.withColumn(
            col,
            F.last(col, ignorenulls=True).over(w_bfill)
        )

In [0]:
display(get_null_info(df_weather_1y_hourly_fm15_ffill_bfill))

In [0]:
# Filter on REPORT_TYPE FM-15 (Routine weather report). This is the most complete and consistent report type, ideal for linking to individual flights
df_weather_1y_hourly_ffill_bfill_fm15 = df_weather_1y_hourly_ffill_bfill.filter(F.col("REPORT_TYPE") == "FM-15")


# Checkpoint preprocessed 1y weather data

In [0]:
# df_weather_1y_hourly_ffill_bfill = df_weather_1y_hourly_ffill_bfill.dropna()
df_weather_1y_hourly_ffill_bfill.write.parquet(f"{folder_path}/df_weather_1y_hourly_ffill_bfill.parquet")

In [0]:
temp_df = spark.read.parquet(f"{folder_path}/df_weather_1y_hourly_ffill_bfill.parquet")
display(temp_df)

In [0]:
display(get_null_info(temp_df))
get_shape(temp_df)

# Size in rows/columns, GB of Airport (Stations) Dataset

In [0]:
df_stations = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/")
df_stations.printSchema()
print(f"Shape of Stations with Neighbors dataset: {get_shape(df_stations)}")

total_size_stations_gb = get_size(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/") / (1024**3)
print(f"Size of Stations with Neighbors dataset: {total_size_stations_gb:.2f} GB")

# Size in rows/columns, GB of Airport Codes Dataset

In [0]:
df_airport_codes = spark.read.format("csv").option("header","true").load('dbfs:/mnt/mids-w261/airport-codes_csv.csv')
df_airport_codes.printSchema()
print(f"Shape of Airport Codes dataset: {get_shape(df_airport_codes)}")

total_size_airport_codes_gb = get_size('dbfs:/mnt/mids-w261/airport-codes_csv.csv') / (1024**3)
print(f"Size of Airport Codes dataset: {total_size_airport_codes_gb} GB")

# Size in rows/columns, GB of Flights Dataset

In [0]:
df_flights_1y = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_1y/")
df_flights_full = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data/")

print(f"Shape of 1 year flights dataset: {get_shape(df_flights_1y)}")
print(f"Shape of full flights dataset: {get_shape(df_flights_full)}")

total_size_1y_gb = get_size(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_1y/") / (1024**3)
print(f"Size of 1 year flights dataset: {total_size_1y_gb:.2f} GB")

total_size_full_gb = get_size(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data/") / (1024**3)
print(f"Size of full flights dataset: {total_size_full_gb:.2f} GB")

# Deduplicate flights data

In [0]:
df_flights_dedup_1y = df_flights_1y.dropDuplicates()
df_flights_dedup_full = df_flights_full.dropDuplicates()

print(f"Shape of 1 year flights dataset after deduplication: {get_shape(df_flights_dedup_1y)}")
print(f"Shape of full flights dataset after deduplication: {get_shape(df_flights_dedup_full)}")

df_flights_dedup_1y.write.parquet(f"{folder_path}/df_flights_dedup_1y.parquet")
total_size_1y_gb = get_size(f"{folder_path}/df_flights_dedup_1y.parquet") / (1024**3)
print(f"Size of 1 year flights dataset after deduplication: {total_size_1y_gb:.2f} GB")

df_flights_dedup_full.write.parquet(f"{folder_path}/df_flights_dedup_full.parquet")
total_size_full_gb = get_size(f"{folder_path}/df_flights_dedup_full.parquet") / (1024**3)
print(f"Size of full flights dataset after deduplication: {total_size_full_gb:.2f} GB")

# Size in rows/columns, GB of OTPW Dataset

In [0]:
# TBD

In [0]:
display(df_weather_full.select('YEAR').dropDuplicates())

In [0]:
df_weather_full.columns

In [0]:
# Create checkpoint folder
section = "03"
number = "02"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"

df_weather_hourly = spark.read.parquet(f"{folder_path}/df_weather_hourly.parquet")
display(df_weather_hourly)

# Size in terms of rows and columns of 1y Weather Dataset

In [0]:
# 1y weather data
df_weather_1y = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_1y/")

# Num rows/cols
print(get_shape(df_weather_1y))

In [0]:
path = f"{folder_path}/df_weather_hourly.parquet"

# Get file info
files = dbutils.fs.ls(path)

# Sum file sizes (in bytes)
total_size_bytes = sum(f.size for f in files)

# Convert to GB
total_size_gb = total_size_bytes / (1024**3)

print(f"Size: {total_size_gb:.2f} GB")

In [0]:
display(df_weather_1y.select(['Station']).distinct())